In [2]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", None)  # show all cols
pd.set_option("display.max_colwidth", None)  # show full width of showing cols
pd.set_option(
    "display.expand_frame_repr", False
)  # print cols side by side as it's supposed to be


ODIS_DUCKDB_FILE = "odis.duckdb"
PCC_DUCKDB_FILE = "dev.duckdb"

con = duckdb.connect(database=PCC_DUCKDB_FILE, read_only=True)
con.sql(f"ATTACH '{ODIS_DUCKDB_FILE}' AS odis;")


In [3]:
query_catnat = """
SELECT *
  FROM dev.main.catnat_gaspar
  ORDER BY cod_commune ASC;
"""

cat_nat_2000 = con.sql(query_catnat)
cat_nat_2000_df = cat_nat_2000.df()

query_odis3 = """SELECT * FROM odis.gold_gold_typologies_territoires
ORDER BY codgeo ASC;"""
odis_topology = con.sql(query_odis3).df()


In [4]:
con = duckdb.connect()

query = open(
    r"C:\\Users\\marin\\OneDrive\Documentos\\GitHub\\14_PrixChangementClimatique\\data\\dbt_pipeline\\models\bronze\\primes_assurances_communes.sql"
).read()

assurance_df = con.execute(query).fetchdf()

print(assurance_df.head())

   annee           siret code_departement  code_insee  code_region  compte  solde_debiteur  solde_crediteur code_geo
0   2021  21170035600010              017          35           75    6161         1618.92              0.0    01735
1   2021  21020032500012              002          32           32    6168          866.22              0.0    00232
2   2021  21550504100016              055         504           44    6161         5138.16              0.0   055504
3   2021  21570058400208              057          58           44    6161        46737.03              0.0    05758
4   2021  21860182100018              086         182           75    6168          694.00              0.0   086182


In [5]:
# informacion de comunas y riesgos
data_chomage = pd.read_csv(
    r"C:\Users\marin\OneDrive\Documentos\GitHub\14_PrixChangementClimatique\data\dbt_pipeline\pipeline_inputs\taux_chomage_communes.csv"
)

In [6]:
from datetime import datetime

climate_data = pd.DataFrame(
    {
        "lib_commune": cat_nat_2000_df["lib_commune"],
        "cod_commune": cat_nat_2000_df["cod_commune"],
        "climate_risk": cat_nat_2000_df["lib_risque_jo"],
        "durée événement": pd.to_datetime(cat_nat_2000_df["dat_fin"], format="mixed")
        - pd.to_datetime(cat_nat_2000_df["dat_deb"], format="mixed"),
        "cat_nat_recognition": cat_nat_2000_df["dat_pub_jo"],
        "recognition_time": pd.to_datetime(
            cat_nat_2000_df["dat_pub_jo"], format="mixed"
        )
        - pd.to_datetime(cat_nat_2000_df["dat_fin"], format="mixed"),
    }
)
climate_df = (
    climate_data.groupby(["cod_commune", "lib_commune", "climate_risk"])
    .agg(
        duree_event_mean=("durée événement", "mean"),
        duree_event_max=("durée événement", "max"),
        nb_events=("durée événement", "count"),
        time_recognition_mean=("recognition_time", "mean"),
        time_recognition_max=("recognition_time", "max"),
    )
    .reset_index()
)

In [7]:
data_chomage.head()  # donnes T3_2025_proxy_commune codgeo

,code_region,region,code_departement,departement,code_commune,commune,codgeo,nombre_de_demandeurs_d_emploi,p23_pop,ratio_ABC_pop_commune,ratio_ABC_pop_dep,T3_2025_departement,T3_2025_proxy_commune
0,84,Auvergne-Rhône-Alpes,01,Ain,01006,Ambléon,01006,10,115.0,0.086957,0.065644,5.7,7.55
1,84,Auvergne-Rhône-Alpes,01,Ain,01008,Ambutrix,01008,35,760.0,0.046053,0.065644,5.7,4.00
2,84,Auvergne-Rhône-Alpes,01,Ain,01011,Apremont,01011,25,415.0,0.060241,0.065644,5.7,5.23
3,84,Auvergne-Rhône-Alpes,01,Ain,01012,Aranc,01012,20,331.0,0.060423,0.065644,5.7,5.25
4,84,Auvergne-Rhône-Alpes,01,Ain,01013,Arandas,01013,10,138.0,0.072464,0.065644,5.7,6.29


In [8]:
climate_data.head()  # durée événement	recognition_time climate_risk cod_commune

,lib_commune,cod_commune,climate_risk,durée événement,cat_nat_recognition,recognition_time
0,L'Abergement-Clémenciat,01001,Inondations et/ou Coulées de Boue,0 days,1984-10-18,119 days
1,L'Abergement-Clémenciat,01001,Inondations et/ou Coulées de Boue,1 days,2009-05-21,103 days
2,L'Abergement-de-Varey,01002,Inondations et/ou Coulées de Boue,5 days,1990-03-23,33 days
3,L'Abergement-de-Varey,01002,Inondations et/ou Coulées de Boue,0 days,2000-11-15,154 days
4,Ambérieu-en-Bugey,01004,Inondations et/ou Coulées de Boue,3 days,1992-03-29,96 days


In [9]:
assurance_df.head()  # code_geo solde_debiteur solde_crediteur

,annee,siret,code_departement,code_insee,code_region,compte,solde_debiteur,solde_crediteur,code_geo
0,2021,21170035600010,017,35,75,6161,1618.92,0.0,01735
1,2021,21020032500012,002,32,32,6168,866.22,0.0,00232
2,2021,21550504100016,055,504,44,6161,5138.16,0.0,055504
3,2021,21570058400208,057,58,44,6161,46737.03,0.0,05758
4,2021,21860182100018,086,182,75,6168,694.00,0.0,086182


## Visualisations


In [10]:
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
# Codgeo VS solde_debiteur


In [ ]:
# cod_commun Vs recognition_time
plt.figure(figsize=(12, 6))
sns.scatterplot(data=climate_df, x="cod_commune", y="time_recognition_mean")

In [ ]:
# recognition_time Vs climate_risk